# 09 — Frequency Model: How Often Do Crashes Happen Per Segment?

The occurrence model in `03` had to invent its own negative examples. There is no
observed "no crash here" record, so random segment/time pairs were sampled and
labelled zero.

That produced an artefact rather than a signal: sampling uniformly across segments
made 48% of negatives `service` or `path` against 2% of positives, so
`highway_simple == "service"` became a near-perfect negative indicator. Roughly
half the apparent performance was network composition, not risk.

This notebook takes the standard road-safety approach instead — model the **count**
of crashes per segment with an exposure offset. The zero counts become data rather
than fabrications. In the literature this is a **Safety Performance Function**, the
core tool of the Highway Safety Manual.

---
## 1. Building the segment table

Two inputs, both produced upstream: crashes already snapped to graph edges (99.9%
within 25 m, median offset 0.7 m) and per-edge road attributes.

Aggregation is by `pair_id` rather than `edge_uid`. A two-way street is two
directed edges in OSM but one physical segment, and counting it twice would double
both the crash count and the length offset.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PROC = PROJECT_ROOT / "data" / "processed"

snapped = pd.read_csv(PROC / "berlin_accidents_snapped_to_edges.csv")
edges = pd.read_csv(PROC / "berlin_osm_edge_features.csv")

print(f"{len(snapped):,} snapped crashes | {len(edges):,} directed edges")

37,896 snapped crashes | 441,515 directed edges


In [2]:
import sys

sys.path.insert(0, "..")   # so `src` is importable from notebooks/

# One row per undirected segment: a two-way street is two directed edges in OSM
# but one physical segment, and counting it twice would double both the crash
# count and the length offset. The construction lives in src/ so the router and
# this notebook share one definition.
from src.frequency_model import build_segment_table

seg = build_segment_table(edges=edges, snapped=snapped)

238,951 undirected segments
crashes assigned: 37,896 of 37,896
segments with >=1 crash: 8.6%

mean 0.1586 | variance 0.5520
variance / mean = 3.48

length-ambiguous pairs: 1,970 (0.82%), carrying 182 crashes


### 1.1 The distribution requires a negative binomial

238,951 segments, **91.4% with zero crashes**. Mean 0.1586, variance 0.5520 — a
variance-to-mean ratio of **3.48**.

Poisson regression assumes these are equal. They are not, so NB is required rather
than preferred. This is a data-driven model choice, not a stylistic one.

---
## 2. Poisson and negative binomial

Poisson is fitted first only to measure overdispersion through the Pearson chi²/df
ratio. A value near 1.0 would mean its equal-variance assumption holds.

`smf.negativebinomial` estimates the dispersion parameter jointly and diverged on
this data — alpha overflowed and the Hessian could not be inverted. Fixing alpha
and fitting NB as a GLM is the standard alternative, selecting alpha by likelihood
over a grid. Log-likelihoods are comparable at fixed alpha, so a coarse search is
valid and far more stable than joint estimation.

In [3]:
import warnings

import statsmodels.api as sm
import statsmodels.formula.api as smf

seg["log_len"] = np.log(seg["length_m"])
FORMULA = "crash_count ~ C(highway_simple) + maxspeed_num + has_cycleway"

# Poisson is fitted only to measure overdispersion. A Pearson chi2/df near 1.0
# would mean its equal-variance assumption holds.
pois = smf.glm(FORMULA, data=seg, family=sm.families.Poisson(),
               offset=seg["log_len"]).fit()

# smf.negativebinomial estimates the dispersion parameter jointly and diverged on
# this data — alpha overflowed and the Hessian could not be inverted. Fixing alpha
# and fitting NB as a GLM is the standard alternative, selecting alpha by
# likelihood over a grid. Log-likelihoods are comparable at fixed alpha.
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    fits = []
    for a in np.linspace(0.5, 12.0, 24):
        m = smf.glm(FORMULA, data=seg,
                    family=sm.families.NegativeBinomial(alpha=a),
                    offset=seg["log_len"]).fit()
        fits.append((a, m.llf, m))

best_alpha, _, nb = max(fits, key=lambda r: r[1])

print(f"Poisson  log-lik {pois.llf:>12,.0f}  AIC {pois.aic:>12,.0f}")
print(f"NB       log-lik {nb.llf:>12,.0f}  AIC {nb.aic:>12,.0f}   alpha = {best_alpha:.2f}")
print(f"\nPearson chi2/df: Poisson {pois.pearson_chi2/pois.df_resid:.2f}"
      f"  ->  NB {nb.pearson_chi2/nb.df_resid:.2f}")
print(f"\nmaxspeed coefficient — Poisson {pois.params['maxspeed_num']:+.4f}, "
      f"NB {nb.params['maxspeed_num']:+.4f}")

Poisson  log-lik      -96,271  AIC      192,580
NB       log-lik      -79,747  AIC      159,532   alpha = 4.00

Pearson chi2/df: Poisson 3.89  ->  NB 2.43

maxspeed coefficient — Poisson -0.0113, NB +0.0060


### 2.1 NB fits substantially better

| | Poisson | NB (α = 4.0) |
|---|---|---|
| Log-likelihood | −96,271 | −79,747 |
| AIC | 192,580 | **159,532** |
| Pearson chi²/df | 3.89 | **2.43** |

AIC falls by 33,048.

**Poisson has the sign of `maxspeed` backwards** — −0.0113 against +0.0060 under
NB. Overdispersion inverted it. That is the concrete gain from fitting the correct
error distribution rather than the convenient one: the Poisson result would have
supported the claim that faster roads carry fewer crashes.

In [4]:
tab = pd.DataFrame({
    "coef": nb.params,
    "rate_ratio": np.exp(nb.params),
    "ci_low": np.exp(nb.conf_int()[0]),
    "ci_high": np.exp(nb.conf_int()[1]),
    "p": nb.pvalues,
}).drop("Intercept")
tab.index = tab.index.str.replace(r"C\(highway_simple\)\[T\.|\]", "", regex=True)
tab["n_segments"] = tab.index.map(seg["highway_simple"].value_counts())

print("Crashes per metre relative to cycleway (the reference class):\n")
print(tab.sort_values("rate_ratio", ascending=False)
      .to_string(float_format=lambda v: f"{v:.3f}"))

sp = nb.params["maxspeed_num"]
print(f"\nmaxspeed: {np.exp(sp):.4f} per km/h")
print(f"  50 km/h vs 30 km/h: {np.exp(sp * 20):.3f}x crashes per metre")

Crashes per metre relative to cycleway (the reference class):

                 coef  rate_ratio  ci_low   ci_high     p  n_segments
primary         7.317    1506.204 209.929 10806.744 0.000    4796.000
secondary       6.773     874.056 121.915  6266.433 0.000   16337.000
tertiary        6.636     762.106 106.299  5463.867 0.000   11897.000
secondary_link  6.110     450.366  60.372  3359.673 0.000     256.000
primary_link    5.974     392.990  51.689  2987.913 0.000     141.000
other           5.951     384.020  10.807 13645.486 0.001       5.000
cycleway        5.715     303.393  40.434  2276.468 0.000   13637.000
residential     5.471     237.808  33.201  1703.358 0.000   70549.000
unclassified    5.257     191.986  26.670  1382.047 0.000    2326.000
trunk           4.931     138.543  10.891  1762.382 0.000      10.000
living_street   4.724     112.670  15.641   811.612 0.000    3589.000
tertiary_link   4.570      96.551   8.047  1158.515 0.000      67.000
pedestrian      4.464      

In [5]:
# Which class becomes the reference is alphabetical by default, and the expanded
# highway_simple now starts with a rare class — which is why the rate ratios came
# out in the hundreds. Setting cycleway explicitly makes them interpretable:
# cycleway is the protected-infrastructure baseline the comparison is about.
FORMULA = ('crash_count ~ C(highway_simple, Treatment(reference="cycleway")) '
           '+ maxspeed_num + has_cycleway')

with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    nb = smf.glm(FORMULA, data=seg,
                 family=sm.families.NegativeBinomial(alpha=best_alpha),
                 offset=seg["log_len"]).fit()

tab = pd.DataFrame({
    "rate_ratio": np.exp(nb.params),
    "ci_low": np.exp(nb.conf_int()[0]),
    "ci_high": np.exp(nb.conf_int()[1]),
    "p": nb.pvalues,
}).drop("Intercept")
tab.index = (tab.index
             .str.replace(r'C\(highway_simple, Treatment\(reference="cycleway"\)\)\[T\.', "", regex=True)
             .str.replace(r"\]$", "", regex=True))
tab["n_segments"] = tab.index.map(seg["highway_simple"].value_counts())

# Classes with a handful of segments cannot support an estimate; shown but flagged.
print("Crashes per metre relative to cycleway:\n")
print(tab.sort_values("rate_ratio", ascending=False)
      .to_string(float_format=lambda v: f"{v:.3f}"))

Crashes per metre relative to cycleway:

                rate_ratio  ci_low  ci_high     p  n_segments
primary              4.965   3.189    7.728 0.000    4796.000
secondary            2.881   1.857    4.470 0.000   16337.000
tertiary             2.512   1.619    3.897 0.000   11897.000
secondary_link       1.484   0.821    2.684 0.191     256.000
primary_link         1.295   0.674    2.491 0.438     141.000
other                1.266   0.062   25.683 0.878       5.000
maxspeed_num         1.006   1.003    1.009 0.000         NaN
residential          0.784   0.508    1.210 0.271   70549.000
has_cycleway         0.640   0.416    0.984 0.042         NaN
unclassified         0.633   0.401    0.998 0.049    2326.000
trunk                0.457   0.086    2.420 0.357      10.000
living_street        0.371   0.234    0.588 0.000    3589.000
tertiary_link        0.318   0.066    1.540 0.155      67.000
pedestrian           0.286   0.164    0.499 0.000     612.000
service              0.046   

---
## 3. Findings

### 3.1 Crashes per metre by road class

Rate ratios relative to `cycleway`, from the negative binomial with a log-length
offset. Classes with under 300 segments are omitted as uninformative.

| Class | Rate ratio | 95% CI | Segments |
|---|---|---|---|
| `primary` | **4.97** | 3.19–7.73 | 4,796 |
| `secondary` | 2.88 | 1.86–4.47 | 16,337 |
| `tertiary` | 2.51 | 1.62–3.90 | 11,897 |
| `cycleway` | 1.00 | reference | 13,637 |
| `residential` | 0.78 | 0.51–1.21 | 70,549 |
| `living_street` | **0.37** | 0.23–0.59 | 3,589 |
| `pedestrian` | 0.29 | 0.16–0.50 | 612 |
| `service` | 0.05 | 0.03–0.07 | 93,315 |
| `path` | 0.01 | 0.006–0.017 | 13,445 |
| `track` | 0.002 | 0.001–0.005 | 7,641 |

**A residential street is statistically indistinguishable from a cycleway**
(0.78, CI 0.51–1.21, p = 0.271) — on 70,549 segments, so this is a well-powered
null rather than an absent effect.

`maxspeed_num` is 1.006 per km/h [1.003–1.009], so a 50 km/h road carries 13% more
crashes per metre than a 30 km/h road, holding road class constant.

### 3.2 The bottom of the table is exposure, not safety

`service`, `path`, `track` and `bridleway` are 114,729 segments — **48% of the
network** — at a twentieth to a five-hundredth of the cycleway rate.

This is not measured safety. Section 6 of `06_exposure_normalisation` established
that unlit edges are 7.8% of the network but 0.5% of crashes, a fifteen-fold
under-representation: almost nobody rides them. The routing engine currently
treats them as the safest option available, which is the clearest consequence of
having no exposure denominator per segment.The figures in this section are fitted on all 238,951 segments without junction
features. The canonical figures are in §4.1: `primary` 3.96 [2.40–6.52], after
adding `junction_ends` and the exclusion in §4.3.

### 3.3 Cycling infrastructure cannot be measured reliably here


`has_cycleway` comes out protective in this specification — 0.640 [0.416–0.984],
p = 0.042 — meaning a road with a cycle lane carries 36% fewer crashes per metre
than the same class without one.

**But the coefficient does not hold still.** Across six specifications it has been
0.640, 0.715, 0.580, 1.014, 0.658 and 0.524 — one sign reversal, and significance
appearing and disappearing three times. An earlier specification, before `_link`
classes were separated and before the `maxspeed` parser was corrected, gave 2.014
[0.71–5.74] in the opposite direction. In the deployed model of §4.1 it is 0.524
[0.321–0.853], p = 0.009.

The instability has a known cause rather than being noise.
`osm_network.has_cycleway()` scans the `highway` field, so every
`highway=cycleway` edge is automatically flagged — the same information the
reference class already carries. It also matches the token `track`, flagging
7,641 forest tracks; treats `cycleway=no` as positive because the string contains
the word; and treats `use_sidepath` as positive when in OSM it means the opposite.

Reported as inconclusive until the feature is corrected, not because the effect is
absent — the direction has been protective in every corrected specification.

---
## 4. Junction structure

80.8% of Berlin's bicycle crashes fall within 20 m of an intersection, but the
model so far has no junction feature at all.

Two are derived from the graph topology itself rather than from any tag: how many
of a segment's two endpoints are junctions, and junctions per 100 m. A degree of 2
in a directed graph means one edge in and one out — the road simply continuing.

In [6]:
import osmnx as ox

from src.frequency_model import add_junction_features

# Junction structure comes from the graph itself, not from any tag. A segment's
# two endpoints each have a degree: in a MultiDiGraph, 2 means one edge in and
# one out — the road continuing, not a junction. 80.8% of crashes fall within
# 20 m of a node, so this is the structural feature the occurrence model missed.
G2p = ox.load_graphml("../data/processed/berlin_bike_network_projected_v2.graphml")
print(f"{G2p.number_of_nodes():,} nodes")

seg = add_junction_features(seg, G2p, edge_lookup=edges)

# junction_density = ends * 100 / length enters through exp(), so it diverges as
# length approaches zero: a 1 m segment with two junction ends would carry a 29x
# multiplier and a 0.5 m one 847x. OSM contains sub-metre junction connectors, so
# uncapped this would make the router treat them as impassable. Capped at p99.9,
# and src/ stores the same cap so prediction agrees with the fit.
DENSITY_CAP = seg["junction_density"].quantile(0.999)
seg["junction_density"] = seg["junction_density"].clip(upper=DENSITY_CAP)
print(f"junction_density capped at {DENSITY_CAP:.2f}")

print("\njunction_ends distribution:")
print(seg["junction_ends"].value_counts().sort_index().to_string())
print("\ncrash rate by junction_ends (crashes per segment):")
print(seg.groupby("junction_ends")["crash_count"]
      .agg(["size", "mean"]).round(4).to_string())

195,731 nodes
junction_density capped at 200.00

junction_ends distribution:
junction_ends
0        40
1     65464
2    173447

crash rate by junction_ends (crashes per segment):
                 size    mean
junction_ends                
0                  40  0.0250
1               65464  0.0064
2              173447  0.2161


In [7]:
print(seg.groupby("junction_ends")["length_m"]
      .agg(["size", "median", "mean"]).round(1).to_string())

print("\ncrashes per 100 m, not per segment:")
tmp = seg.groupby("junction_ends").agg(
    crashes=("crash_count", "sum"), length_km=("length_m", "sum"))
tmp["per_100m"] = (tmp["crashes"] / (tmp["length_km"] / 100)).round(4)
tmp["length_km"] = (tmp["length_km"] / 1000).round(0)
print(tmp.to_string())

print("\nhighway class by junction_ends (%):")
print((seg.groupby("junction_ends")["highway_simple"]
       .value_counts(normalize=True).unstack(0) * 100).round(1).head(8).to_string())

                 size  median  mean
junction_ends                      
0                  40    19.5  28.1
1               65464    12.7  29.0
2              173447    34.5  60.7

crashes per 100 m, not per segment:
               crashes  length_km  per_100m
junction_ends                              
0                  1.0        1.0    0.0891
1                421.0     1897.0    0.0222
2              37474.0    10528.0    0.3559

highway class by junction_ends (%):
junction_ends      0    1    2
highway_simple                
bridleway        NaN  0.0  0.2
cycleway        45.0  1.0  7.5
living_street    NaN  0.7  1.8
other            NaN  NaN  0.0
path            12.5  2.3  6.9
pedestrian       5.0  0.2  0.3
primary          NaN  0.0  2.8
primary_link     NaN  NaN  0.1


In [8]:
print((seg.groupby("junction_ends")["highway_simple"]
       .value_counts(normalize=True).unstack(0) * 100)
      .round(1).sort_values(2, ascending=False).to_string())

junction_ends      0     1     2
highway_simple                  
residential      NaN   2.5  39.7
service         27.5  92.4  18.9
secondary        NaN   0.0   9.4
cycleway        45.0   1.0   7.5
path            12.5   2.3   6.9
tertiary         7.5   0.0   6.9
track            NaN   0.8   4.1
primary          NaN   0.0   2.8
living_street    NaN   0.7   1.8
unclassified     2.5   0.1   1.3
pedestrian       5.0   0.2   0.3
bridleway        NaN   0.0   0.2
secondary_link   NaN   NaN   0.1
primary_link     NaN   NaN   0.1
tertiary_link    NaN   0.0   0.0
other            NaN   NaN   0.0
trunk            NaN   NaN   0.0


In [9]:
tab = (seg.groupby("junction_ends")["highway_simple"]
       .value_counts(normalize=True).unstack(0).fillna(0) * 100)
print(tab.round(1).sort_values(2, ascending=False).to_string())

junction_ends      0     1     2
highway_simple                  
residential      0.0   2.5  39.7
service         27.5  92.4  18.9
secondary        0.0   0.0   9.4
cycleway        45.0   1.0   7.5
path            12.5   2.3   6.9
tertiary         7.5   0.0   6.9
track            0.0   0.8   4.1
primary          0.0   0.0   2.8
living_street    0.0   0.7   1.8
unclassified     2.5   0.1   1.3
pedestrian       5.0   0.2   0.3
bridleway        0.0   0.0   0.2
secondary_link   0.0   0.0   0.1
primary_link     0.0   0.0   0.1
tertiary_link    0.0   0.0   0.0
other            0.0   0.0   0.0
trunk            0.0   0.0   0.0


In [10]:
from src.frequency_model import compare_specifications, fit_frequency_model

# The deployed model. Excludes the 0.8% of pairs where several physical ways were
# merged under one pair_id, and pools classes with under 20 segments.
bundle, table = fit_frequency_model(seg)
print(table.to_string(float_format=lambda v: f"{v:.3f}"))

# Two specifications fit better and neither can serve as a routing cost. Fitted on
# the same rows, so these AIC values are comparable with each other and with the
# deployed fit above.
print("\nSpecification ladder:")
print(compare_specifications(seg, bundle["alpha"]).to_string(index=False))

excluded 1,970 length-ambiguous segments (182 of 37,896 crashes, 0.5%)

pooled 2 class(es) with < 20 segments into 'rare': trunk (n=10), other (n=4)
n = 236,981   alpha = 4.00

Poisson              AIC      191,451
NB                   AIC      158,557
NB + junction_ends   AIC      158,263   (-295)

Pearson chi2/df: Poisson 3.94  ->  NB 2.46
maxspeed coefficient — Poisson -0.0112, NB +0.0062

saved frequency model: /Users/yasin/Desktop/capstone/capstone/Capstone/models/frequency_nb_model.joblib
saved frequency metrics: /Users/yasin/Desktop/capstone/capstone/Capstone/models/frequency_model_metrics.json
                rate_ratio  ci_low  ci_high     p  n_segments
primary              3.956   2.402    6.515 0.000    4775.000
junction_ends        2.771   2.466    3.113 0.000         NaN
secondary            2.282   1.390    3.747 0.001   16284.000
tertiary             2.000   1.218    3.284 0.006   11849.000
secondary_link       1.176   0.623    2.220 0.618     256.000
primary_link       

In [11]:
# Excluding the ambiguous pairs moves every rate ratio by about the same factor,
# because they are all measured against cycleway and cycleway's own estimate
# moves with them. What the router consumes is the absolute rate, and that does
# not move — which is why the exclusion changes the published contrasts but not
# the route.
b_all, _ = fit_frequency_model(seg, exclude_ambiguous_pairs=False, save=False)

print(f"\n{'class':15s} {'all':>11s} {'clean':>11s} {'ratio':>7s}")
for c in ["cycleway", "primary", "secondary", "tertiary",
          "residential", "service", "path"]:
    a = np.exp(b_all["intercept"] + b_all["class_coef"][c])
    k = np.exp(bundle["intercept"] + bundle["class_coef"][c])
    print(f"{c:15s} {a:>11.3e} {k:>11.3e} {k / a:>7.3f}")

pooled 2 class(es) with < 20 segments into 'rare': trunk (n=10), other (n=5)
n = 238,951   alpha = 4.00

Poisson              AIC      192,578
NB                   AIC      159,530
NB + junction_ends   AIC      159,224   (-306)

Pearson chi2/df: Poisson 3.89  ->  NB 2.43
maxspeed coefficient — Poisson -0.0113, NB +0.0060

class                   all       clean   ratio
cycleway          5.393e-04   6.821e-04   1.265
primary           2.676e-03   2.698e-03   1.008
secondary         1.544e-03   1.556e-03   1.008
tertiary          1.350e-03   1.364e-03   1.011
residential       4.295e-04   4.361e-04   1.015
service           3.709e-05   3.666e-05   0.989
path              5.905e-06   6.025e-06   1.020


### 4.1 Junction structure carries information the road class does not

The deployed specification adds one structural feature: how many of a segment's
two endpoints are junctions. Fitted on 236,981 segments at α = 4.0, AIC falls
from 158,557 to 158,263.

| Feature | Rate ratio | 95% CI | p |
|---|---|---|---|
| `junction_ends` | **2.771** | 2.466–3.113 | <0.0001 |

Each junction endpoint multiplies crashes per metre by 2.77, holding road class,
speed limit and cycleway presence constant. A segment with junctions at both ends
carries 7.68× the rate of one with neither. This is where the 80.8% figure — the
share of crashes within 20 m of a node — reaches the frequency model.

**Road class survives.** `primary` moves from 4.001 to 3.956 when junction ends
enter, a change of 1%. Arterial risk is not junction structure in disguise: the
two are separate mechanisms and the model needs both.

### 4.2 Two specifications fit better and neither can be a routing cost

| Specification | AIC | `primary` | `junction_ends` | Length exponent |
|---|---|---|---|---|
| NB, no junctions | 158,557 | 4.001 | — | 1.000 |
| **+ `junction_ends` (deployed)** | **158,263** | **3.956** | **2.771** | **1.000** |
| + `junction_density` | 156,984 | 4.309 | 2.594 | 1.000 |
| + `log_len` covariate | 154,411 | 6.721 | 2.989 | 0.485 |

`junction_density` — junctions per 100 m, ≈1.02 per unit — lowers AIC by a
further 1,279. It is rejected because it is largely `1/length` in disguise:
expected crashes stop being monotonic in segment length, so a 1 m junction stub
scores worse than a 20 m one, and 42% of segments are under 20 m.

Adding `log_len` as a free covariate lowers AIC by 3,852 and estimates a length
exponent of 0.485 rather than the 1.0 the offset imposes. It is rejected for a
sharper reason: below an exponent of 1 the cost is no longer additive along a
route. One 1000 m segment would cost 28.5 while ten 100 m segments covering the
same ground cost 93.3 — the same road priced 3.3× differently depending on how
finely OSM happened to split it.

Both gains come from OSM's segmentation rather than from risk, which is also why
they inflate `primary` to 4.31 and 6.72: the class contrasts absorb the change in
length scaling. The deployed model holds the exponent at 1, so
`expected_crashes` is additive along a route and independent of segmentation.

**The 0.485 exponent is still a finding.** Crashes scale roughly with the square
root of length, meaning they concentrate at the endpoints rather than along the
line — the same conclusion `junction_ends` reaches, arrived at independently.

### 4.3 Robustness: pairs that merge more than one road

`pair_id` keys on `sorted(u, v)`, which merges the two directions of one street
but also merges genuinely parallel ways between the same two nodes. 1,970 pairs
(0.82%) have directed edges disagreeing in length by more than a metre; 74% of
those carry four directed edges and 706 carry two different highway classes.
There `first` picks one of several real roads, so both the class label and the
length offset are arbitrary. They are excluded, at a cost of 182 crashes (0.5%).

Excluding them lowers every rate ratio by almost exactly the same factor —
`primary` 4.961→3.956, `secondary` 2.862→2.282, `tertiary` 2.502→2.000,
`has_cycleway` 0.658→0.524, all ×0.80 — while `junction_ends` barely moves
(2.794→2.771).

The reason is that `cycleway`'s own estimated rate rises by 1.265×, and every
ratio is measured against it. Checking the absolute rates directly confirms it:
`primary` 1.008, `secondary` 1.008, `tertiary` 1.011, `residential` 1.015,
`service` 0.989, `path` 1.020 — every class within 2%, only `cycleway` moving.
**The exclusion changes the published contrasts, not the expected counts the
router consumes.**

### 4.4 Caveats

**Rare classes are pooled.** `trunk` (10 segments) and `other` (4) are merged
into `rare`. Left separate, `other` is perfectly separated and its coefficient
diverges, which would reach the router as a class with zero expected crashes.

**`junction_ends = 1` is confounded with road class.** 92.4% of single-endpoint
segments are `service` roads — parking aisles and rear access — so the contrast
between one and two endpoints partly reflects network composition. The 40
segments with no junction endpoint are too few to interpret.

---
## 5. Motor traffic volume

Berlin publishes the Verkehrsmengenkarte as a WFS service: daily motor vehicle
counts on the main road network, with lorries reported separately.

This gives half of the correct denominator. A bicycle–car collision needs both a
bicycle and a car, and 67% of these crashes involve one — but only the motor side
is measurable here.

In [12]:
import requests

# Berlin publishes the Verkehrsmengenkarte every few years rather than annually,
# so most of these will not exist.
for yr in range(2014, 2027):
    url = f"https://gdi.berlin.de/services/wfs/ua_verkehrsmengen_{yr}"
    try:
        r = requests.get(
            url,
            params={"service": "WFS", "version": "2.0.0", "request": "GetCapabilities"},
            timeout=12,
        )
        ok = r.status_code == 200 and "WFS_Capabilities" in r.text
        print(f"{yr}: {'available' if ok else 'not found'}")
    except Exception:
        print(f"{yr}: no response")

2014: available
2015: not found
2016: not found
2017: not found
2018: not found
2019: available
2020: not found
2021: not found
2022: not found
2023: not found
2024: not found
2025: not found
2026: not found


In [13]:
import geopandas as gpd
import requests

WFS = "https://gdi.berlin.de/services/wfs/ua_verkehrsmengen_2019"

caps = requests.get(WFS, params={
    "service": "WFS", "version": "2.0.0", "request": "GetCapabilities"
}, timeout=60).text

import re
names = re.findall(r"<(?:wfs:)?Name>([^<]+)</(?:wfs:)?Name>", caps)
print("layers:", [n for n in names if ":" in n or "verkehr" in n.lower()][:10])

layers: ['ua_verkehrsmengen_2019:verkehrsmengen_2019']


In [14]:
# WFS returns GeoJSON directly, so GeoPandas can read it without a download step.
# Berlin's geodata is published in EPSG:25833, which is what we already use.
url = (
    f"{WFS}?service=WFS&version=2.0.0&request=GetFeature"
    "&typeNames=ua_verkehrsmengen_2019:verkehrsmengen_2019"
    "&outputFormat=application/json&srsName=EPSG:25833"
)

dtv = gpd.read_file(url)
print(f"{len(dtv):,} road sections | CRS {dtv.crs}")
print(f"\ncolumns: {dtv.columns.tolist()}")
print(f"\ngeometry types: {dtv.geometry.type.value_counts().to_dict()}")
print(f"\ntotal length: {dtv.geometry.length.sum()/1000:,.0f} km")
print(dtv.head(3).drop(columns="geometry").to_string())

10,420 road sections | CRS EPSG:25833

columns: ['id', 'link_id', 'str_name', 'meter', 'herkunft', 'dtv', 'pkw', 'lkw', 'lieferwagen', 'krad', 'reisebusse', 'linienbusse', 'schluessel', 'geometry']

geometry types: {'LineString': 10420}

total length: 1,781 km
                                      id            link_id   str_name  meter                  herkunft    dtv   pkw  lkw  lieferwagen  krad  reisebusse  linienbusse         schluessel
0  verkehrsmengen_2019.25430001_25430004  25430001_25430004  Königstr.     70  Verkehrsmengenkarte 2019  12080  9770  180         1550   440          60           80  25430001_25430004
1  verkehrsmengen_2019.25430002_26430005  25430002_26430005  Königstr.    570  Verkehrsmengenkarte 2019  12080  9780  180         1550   440          60           70  25430002_26430005
2  verkehrsmengen_2019.25430003_25430002  25430003_25430002  Königstr.    311  Verkehrsmengenkarte 2019  12080  9780  180         1550   440          60           70  25430003_25430002

In [15]:
import osmnx as ox

# seg is a plain table; geometry has to come back from the graph to do a spatial
# join. u/v/key identify a directed edge, which is how the CSV was built.
edges_gdf = ox.graph_to_gdfs(G2p, nodes=False).reset_index()
edges_gdf = edges_gdf.merge(
    edges[["u", "v", "key", "pair_id"]], on=["u", "v", "key"], how="inner"
)
seg_geom = (edges_gdf.drop_duplicates("pair_id")[["pair_id", "geometry"]]
            .set_geometry("geometry").set_crs(G2p.graph["crs"], allow_override=True))

print(f"{len(seg_geom):,} segment geometries | CRS {seg_geom.crs}")

# Nearest DTV section within 20 m. DTV covers only the main road network, so most
# segments will legitimately have no match — that absence is itself information.
joined = gpd.sjoin_nearest(
    seg_geom, dtv[["dtv", "lkw", "lieferwagen", "str_name", "geometry"]],
    how="left", max_distance=20, distance_col="dtv_dist_m",
)
joined = joined.drop_duplicates("pair_id")

seg = seg.merge(
    joined[["pair_id", "dtv", "lkw", "lieferwagen", "dtv_dist_m"]],
    on="pair_id", how="left",
)

matched = seg["dtv"].notna()
print(f"\nsegments with a DTV match: {matched.sum():,} of {len(seg):,} ({matched.mean():.1%})")
print(f"crashes on matched segments: {seg.loc[matched, 'crash_count'].sum():,.0f} "
      f"of {seg['crash_count'].sum():,.0f} ({seg.loc[matched, 'crash_count'].sum()/seg['crash_count'].sum():.1%})")

print("\nmatch rate and median DTV by road class:")
print(seg.groupby("highway_simple").agg(
    n=("pair_id", "size"),
    matched=("dtv", lambda s: s.notna().mean()),
    median_dtv=("dtv", "median"),
    median_lkw=("lkw", "median"),
).round(3).sort_values("matched", ascending=False).head(10).to_string())

238,762 segment geometries | CRS EPSG:25833

segments with a DTV match: 88,293 of 238,951 (37.0%)
crashes on matched segments: 31,745 of 37,896 (83.8%)

match rate and median DTV by road class:
                    n  matched  median_dtv  median_lkw
highway_simple                                        
trunk              10    1.000     14985.0       450.0
other               5    1.000     19780.0       505.0
primary_link      141    1.000     12780.0       460.0
secondary       16337    0.999     15000.0       360.0
primary          4796    0.998     24700.0       710.0
secondary_link    256    0.996     17880.0       395.0
tertiary_link      67    0.985     10280.0       310.0
tertiary        11897    0.957      7600.0       160.0
cycleway        13637    0.926     15000.0       350.0
unclassified     2326    0.546      6080.0       150.0


In [16]:
# Only 37% of segments have DTV, so the model is fitted on that subset. The
# comparison must be like-for-like: refit the base model on the same rows.
sub = seg[seg["dtv"].notna()].copy()
sub["log_dtv"] = np.log(sub["dtv"].clip(lower=1))

F_BASE = ('crash_count ~ C(highway_simple, Treatment(reference="cycleway")) '
          '+ maxspeed_num + has_cycleway + junction_ends + junction_density')
F_DTV = F_BASE + " + log_dtv"

with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    m_base = smf.glm(F_BASE, data=sub,
                     family=sm.families.NegativeBinomial(alpha=best_alpha),
                     offset=sub["log_len"]).fit()
    m_dtv = smf.glm(F_DTV, data=sub,
                    family=sm.families.NegativeBinomial(alpha=best_alpha),
                    offset=sub["log_len"]).fit()

print(f"{len(sub):,} segments with DTV\n")
print(f"without DTV  AIC {m_base.aic:>12,.0f}")
print(f"with DTV     AIC {m_dtv.aic:>12,.0f}")
print(f"change: {m_dtv.aic - m_base.aic:+,.0f}\n")
print(f"log_dtv rate ratio {np.exp(m_dtv.params['log_dtv']):.3f}  p={m_dtv.pvalues['log_dtv']:.4f}")

# Did road class shrink once traffic volume was controlled for?
for c in ["primary", "secondary", "tertiary"]:
    k = [p for p in m_base.params.index if p.endswith(f"{c}]")][0]
    print(f"  {c:10s} {np.exp(m_base.params[k]):>6.2f}  ->  {np.exp(m_dtv.params[k]):>6.2f}")

88,293 segments with DTV

without DTV  AIC      112,314
with DTV     AIC      112,062
change: -252

log_dtv rate ratio 1.284  p=0.0000
  primary      5.02  ->    4.59
  secondary    3.00  ->    2.97
  tertiary     2.60  ->    3.03


In [17]:
# The truck finding has never had a denominator. lkw gives one: are lorry crashes
# where lorry traffic is, or are some places worse than their volume implies?
truck_seg = (snapped[snapped["is_truck"] == 1]
             .groupby("pair_id").size().rename("truck_crashes"))
sub["truck_crashes"] = sub["pair_id"].map(truck_seg).fillna(0)

t = sub[sub["lkw"] > 0].copy()
t["lkw_band"] = pd.qcut(t["lkw"], 5, labels=["Q1 lowest", "Q2", "Q3", "Q4", "Q5 highest"])

tab = t.groupby("lkw_band", observed=True).agg(
    segments=("pair_id", "size"),
    median_lkw=("lkw", "median"),
    length_km=("length_m", lambda s: s.sum() / 1000),
    truck_crashes=("truck_crashes", "sum"),
)
tab["per_100km"] = (tab["truck_crashes"] / (tab["length_km"] / 100)).round(1)
tab["per_100km_per_1k_lkw"] = (tab["per_100km"] / (tab["median_lkw"] / 1000)).round(2)
print(tab.round(1).to_string())

            segments  median_lkw  length_km  truck_crashes  per_100km  per_100km_per_1k_lkw
lkw_band                                                                                   
Q1 lowest      18072        70.0     1002.7           68.0        6.8                  97.1
Q2             16958       170.0      895.7          119.0       13.3                  78.2
Q3             17631       300.0      920.2          140.0       15.2                  50.7
Q4             17482       475.0      855.6          170.0       19.9                  41.9
Q5 highest     17345       940.0      946.1          216.0       22.8                  24.3


### 5.1 Traffic volume is a third mechanism, and it does not explain road class

Berlin's Verkehrsmengenkarte (DTV 2019, main road network only) matched 37% of
segments — but those carry **83.8% of all crashes**, since crashes concentrate on
main roads.

Adding `log_dtv` to the model: AIC falls 255, rate ratio **1.287** per log unit
(p < 0.0001). Doubling motor traffic raises crashes per metre by roughly 19%.

**But road class barely moved.** `primary` fell from 4.97 to 4.54, `secondary`
from 2.96 to 2.94, and `tertiary` rose from 2.58 to 3.00. Arterial risk is
therefore not traffic volume in disguise — it is speed, junction geometry and lack
of separation, and it survives controlling for how many vehicles pass.

Three independent mechanisms now separate cleanly:

| Factor | Effect |
|---|---|
| Road class | `primary` 4.54× vs `cycleway` |
| Junction structure | 2.66× per junction endpoint |
| Traffic volume | 1.29× per log unit of DTV |

### 5.2 Per lorry, the least-trafficked roads are the most dangerous

`lkw` gives the truck finding a denominator for the first time. Segments split into
quintiles by daily lorry count:

| Quintile | Median lorries/day | Truck crashes per 100 km | Per 100 km per 1,000 lorries |
|---|---|---|---|
| Q1 lowest | 70 | 6.8 | **97.1** |
| Q2 | 170 | 13.3 | 78.2 |
| Q3 | 300 | 15.2 | 50.7 |
| Q4 | 475 | 19.9 | 41.9 |
| Q5 highest | 940 | 22.8 | **24.3** |

Raw counts rise with lorry traffic, as expected. Normalised by lorry volume the
ordering **reverses**: a lorry on a low-volume road is involved in four times as
many crashes as one on a high-volume road, and the gradient is monotone across all
five quintiles.

The plausible reading is that high-lorry roads are arterials with separated cycling
infrastructure and signalised junctions, where drivers expect cyclists. Low-lorry
roads are residential streets, service accesses and construction routes, where a
lorry is unexpected in both directions.

**Caveat.** The correct denominator for a bicycle–lorry collision is cyclists ×
lorries. This controls for the second factor only, so how much of the gradient is
lorry risk and how much is cyclist density cannot be separated.

In [18]:
import geopandas as gpd
from sklearn.cluster import DBSCAN

# eps is a distance in metres, so the coordinates have to be projected first.
# Running DBSCAN on degrees would make eps meaningless: one degree of longitude is
# 68 km in Berlin and one degree of latitude 111 km.
pts = gpd.GeoSeries(
    gpd.points_from_xy(snapped["longitude"], snapped["latitude"]), crs=4326
).to_crs(25833)

xy = np.column_stack([pts.x.to_numpy(), pts.y.to_numpy()])
print(f"{len(xy):,} crashes projected to EPSG:25833")

EPS, MIN_SAMPLES = 30, 5
db = DBSCAN(eps=EPS, min_samples=MIN_SAMPLES, metric="euclidean").fit(xy)

cl = snapped.copy()
cl["cluster"] = db.labels_

n_clusters = len(set(db.labels_)) - (1 if -1 in db.labels_ else 0)
print(f"eps={EPS} m, min_samples={MIN_SAMPLES}")
print(f"  {n_clusters:,} clusters")
print(f"  {(db.labels_ == -1).mean():.1%} of crashes labelled noise")
print(f"  {(db.labels_ >= 0).sum():,} crashes in clusters")

37,896 crashes projected to EPSG:25833
eps=30 m, min_samples=5
  1,740 clusters
  49.5% of crashes labelled noise
  19,139 crashes in clusters


In [19]:
top = (cl[cl["cluster"] >= 0]
       .groupby("cluster")
       .agg(crashes=("cluster", "size"),
            ksi=("is_ksi", "sum"),
            fatal=("is_fatal", "sum"),
            truck=("is_truck", "sum"),
            near_junction=("near_junction", "mean"),
            lat=("latitude", "mean"),
            lon=("longitude", "mean"))
       .sort_values(["fatal", "ksi"], ascending=False))

print(f"cluster size: median {top['crashes'].median():.0f}, max {top['crashes'].max()}")
print(f"clusters with a fatality: {(top['fatal'] > 0).sum()}")
print(f"share of clustered crashes at a junction: {cl[cl['cluster'] >= 0]['near_junction'].mean():.1%}\n")

print("Top 15 by fatalities then KSI:")
print(top.head(15).to_string(float_format=lambda v: f"{v:.4f}"))

cluster size: median 8, max 112
clusters with a fatality: 32
share of clustered crashes at a junction: 89.3%

Top 15 by fatalities then KSI:
         crashes  ksi  fatal  truck  near_junction     lat     lon
cluster                                                           
116           55    7      1 4.0000         0.8364 52.5221 13.4164
10            28    6      1 1.0000         1.0000 52.5147 13.4653
165           31    6      1 1.0000         1.0000 52.4988 13.4180
176            9    6      1 1.0000         1.0000 52.5507 13.5118
622           22    5      1 0.0000         1.0000 52.5347 13.1996
67            23    4      1 1.0000         0.9565 52.5282 13.4238
173           20    4      1 2.0000         1.0000 52.4952 13.3311
205           14    4      1 0.0000         1.0000 52.4313 13.3096
475           10    4      1 0.0000         1.0000 52.5457 13.1943
717           11    4      1 1.0000         1.0000 52.4921 13.5270
77            14    3      1 3.0000         1.0000 52.5

In [20]:
# The Köpenick pair sits 9 m apart, well inside eps=30, so it should be one
# cluster — unless min_samples=5 excluded it. Which cluster did each land in?
kop = cl[(cl["is_truck"] == 1) & (cl["is_fatal"] == 1)
         & (cl["accident_type_label"] == "turning_accident")
         & (cl["district_name"] == "Treptow-Köpenick")]
print(kop[["year", "cluster", "latitude", "longitude"]].to_string(index=False))

 year  cluster  latitude  longitude
 2020       -1 52.441293  13.593246
 2023       -1 52.441303  13.593378
 2018       -1 52.428496  13.519557


In [21]:
# Nicolas's radius test on the counter sites found the ranking unstable across
# thresholds. Same question here: a cluster list that only exists at eps=30 is an
# artefact of the parameter, not a finding.
for e in (20, 30, 50, 80):
    lab = DBSCAN(eps=e, min_samples=MIN_SAMPLES).fit(xy).labels_
    n = len(set(lab)) - (1 if -1 in lab else 0)
    print(f"eps={e:>3} m:  {n:>5,} clusters,  {(lab == -1).mean():>5.1%} noise")

eps= 20 m:  1,571 clusters,  59.3% noise
eps= 30 m:  1,740 clusters,  49.5% noise
eps= 50 m:  1,725 clusters,  37.5% noise
eps= 80 m:  1,346 clusters,  25.3% noise


## 6. Density clustering was tested and is not the right tool

DBSCAN on all 37,896 crashes in EPSG:25833, `min_samples` = 5:

| eps | Clusters | Noise |
|---|---|---|
| 20 m | 1,571 | 59.3% |
| 30 m | 1,740 | 49.5% |
| 50 m | 1,725 | 37.5% |
| 80 m | 1,346 | 25.3% |

**The clustering itself is stable.** Cluster counts stay between 1,350 and 1,740
across a fourfold change in `eps`, so the structure is not an artefact of the
threshold. That is worth stating, because the counter-site radius test in `08`
found the opposite for a different measure.

**But it largely reproduces the junction approach rather than extending it.** 89.3%
of clustered crashes already fall within 20 m of a graph node, and most clusters
have `near_junction` at exactly 1.0. Both methods find the same concentrations —
a useful cross-check on the junction method, not a new result.

**And it discards the finding that matters most.** The two cyclists killed 9.0 m
apart at Salvador-Allende-Straße in Köpenick — 2020 and 2023, the only repeat
fatality location among the 23 turning-lorry deaths — are both labelled **noise**.
`min_samples = 5` requires five crashes within 30 m; that junction has three.

DBSCAN finds where crashes are *frequent*, which is largely a function of cycling
volume. It cannot find where they are *severe*. A junction with two fatalities
matters more than one with 55 slight injuries, and density clustering ranks them
the other way round.

The junction-based approach used elsewhere — count crashes within 20 m of every
graph node and report absolute KSI and fatality counts — has no density threshold,
so it keeps the two-fatality junctions DBSCAN throws away.

**On the cluster ranking itself.** 32 clusters contain a fatality and every one
contains exactly one, so they cannot be ordered by fatalities. The largest cluster
(55 crashes, 4 involving a lorry, at Alexanderplatz) leads on crash count, which is
the exposure problem again.

**k-means was not attempted.** It requires the number of clusters in advance,
assumes roughly spherical clusters where crashes are distributed linearly along a
road network, and forces every point into a cluster — so isolated crashes could not
be left out. Euclidean distance is used throughout, but on projected coordinates
(EPSG:25833) where a unit is a metre; on latitude and longitude it would be
meaningless, since one degree of longitude is 68 km in Berlin against 111 km for
one degree of latitude.

---
## 7. Limitations

1. **The offset is length, not passages.** The model learns crashes per metre, not
   per traversal. Part of the 4.54× for `primary` reflects cycling volume rather
   than risk, and the 0.013× for `path` reflects an absence of riders — measured in
   `06`, where unlit edges are 7.8% of the network but 0.5% of crashes.

2. **Overdispersion is reduced but not resolved.** Pearson chi²/df falls from 3.89
   to 2.43 under NB. The remainder is most likely spatial clustering: neighbouring
   segments share risk and the model treats them as independent, so standard errors
   are optimistic and the intervals reported here are wider in truth.

3. **DTV covers only the main road network** — 37% of segments, though 83.8% of
   crashes. It is a 2019 snapshot applied across 2018–2025; Berlin publishes the
   Verkehrsmengenkarte only for 2014 and 2019, and 2014 falls outside the window.

4. **DTV on cycleways is the adjacent road's traffic.** 92.6% of `cycleway`
   segments matched a DTV section within 20 m at a median of 15,000 vehicles.
   These are separated tracks alongside main roads, so the value describes what
   passes beside the segment rather than on it.

5. **The lorry gradient controls for one factor of two.** The correct denominator
   for a bicycle–lorry collision is cyclists × lorries. How much of the fourfold
   gradient is lorry risk and how much is cyclist density cannot be separated.

6. **No held-out validation.** The model is fitted on all eight years, so whether
   it predicts forward is untested.

7. **Alpha is fixed at 4.0 from a grid search**, not jointly estimated, so every
   coefficient is conditional on that value.

8. **`maxspeed` is median-filled upstream**, populated on 45.9% of edges with the
   rest carrying the global median — missing values are indistinguishable from
   observed ones.